# LLM reasoning: CoT, self-consistency voting, and measuring emergence


> Previous lectures each treated one part of the Agent: lecture 11 handled memory, lecture 10 turned tasks into writing code, and lecture 14 covers evaluation. They all assume one premise: a single continuation from the model is enough to produce a reasonable next step. Once the task itself is correct only after multi-step reasoning, that premise does not hold, and scaling the model shows little clear gain.
>
> This lecture fills exactly that gap: where reasoning ability comes from, and how to elicit it without changing weights. We first implement a prompt that "lets the model write out intermediate steps", then run an ablation, then implement self-consistency decoding that "tries several times and votes", and finally return to a dispute: some abilities appear suddenly as models grow, and that appearance may be an artifact of the metric.

Consider an elementary-school problem: Roger has 5 tennis balls, then buys 2 cans of 3 balls each. How many balls are there in total?

Under a standard prompt the model reports a number directly, and often guesses wrong: it has to get both "multiply then add" and the final result right in one step, and a single slip makes the whole answer wrong.

Switch to having it write the process: first the newly bought balls, 2×3=6; then add the original 5, 5+6=11; only then write the final answer. Each step sits in the context, so when the model computes the next step it can read the previous result.

The same problem, with "report the answer directly" replaced by "write the steps first, then report the answer", often turns a wrong answer into a right one. This prompt format that writes the reasoning process step by step is **chain-of-thought**.

The CoT paper (Wei et al., 2022) found a contrast on GSM8K, an elementary math set: the same model, the same parameters, and only the exemplars in the prompt changing from "problem—answer" to "problem—reasoning steps—answer", and accuracy rose clearly. That shows some reasoning ability was already in the weights, and what was missing was a suitable prompt format to elicit it.

The first thing this lecture does is to verify that contrast by hand: construct two prompts, one that asks for the answer directly, one that has the model write steps first, then compare the two outputs. Section 1 starts from constructing these two prompts.

This section does one concrete thing: prepare a small problem set, write two prompts, one that asks for the answer directly and one that has the model write intermediate steps first, then see how the model answers each. As noted earlier, step-by-step thinking can elicit ability the model already has; this section checks that claim in code. The principle is one sentence: pretraining wrote language regularities into the weights, reasoning ability included, and the prompt format decides how the model calls those weights. In a real system, the text sent to ChatGPT is the prompt; this section first makes its structure clear.

**Experimental corpus**: the problem set has two demonstration items, both with integer answers. Each of the two exemplar sets has two items; the only difference is whether intermediate reasoning steps are included.

Standard format (⟨problem, answer⟩):

```text
Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: The final answer is 11.
```

Chain-of-thought format (⟨problem, reasoning steps, answer⟩):

```text
Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
A: He first computes the new balls: 2 cans times 3 per can is 6. Adding the original 5, 5 + 6 = 11. The final answer is 11.
```


**Prompt template: concatenating exemplars and the problem**

The two exemplar sets differ by only one part. Extracting the structure gives two templates:

```text
Standard format:   Q: <problem>
                   A: <final answer>

Chain-of-thought:  Q: <problem>
                   A: <reasoning steps>
                   The final answer is <final answer>.
```

Both prompts start with "Q:" and end with "A:". What the model does is continue after the last "A:": under the standard prompt it writes only a number; under the chain-of-thought prompt it first writes intermediate steps, then the final answer.

Concatenating a few exemplars and the problem to be answered into one passage, and letting the model continue, is called few-shot: first give a few examples, then ask a question. One example is too few for the model to tell which parts are the problem and which are the steps; two examples fix the "compute first, then answer" pattern. Exemplars are not used in training; they are only part of the prompt string. What the model copies is the structure, not the content.

This design has three reasons. First, the answer is placed last, so when the model generates the final number, every previous step remains in context and can be cited. Intermediate results are written into the text and need not be stored in the model's internal vectors, which lowers the risk of "forgetting the previous step".

Second, each intermediate step is a fluent sentence of natural language. After writing the previous step, the model tends to continue what it recently wrote, and is less likely to drift into an unrelated topic.

Third, "The final answer is" is a stable closing marker. The parser below relies on it to extract the answer from a long output, and majority voting also depends on the same marker.

In [ ]:
import numpy as np

np.random.seed(42)  # keep the experiment reproducible

# Two demonstration problems; the true answers are 11 and 19
QUESTIONS = [
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?",
    "Mom bought 4 bags of apples, 6 per bag. After eating 5, how many are left?",
]
TRUTH = [11, 19]

# Standard-prompt exemplars: problem and answer only
STANDARD_EXEMPLARS = [
    "Q: Ming has 3 pens and buys 4 more. How many pens in total?\nA: The final answer is 7.",
    "Q: A bookshelf had 9 books; 3 were taken. How many remain?\nA: The final answer is 6.",
]

# Chain-of-thought exemplars: intermediate reasoning steps before the answer
COT_EXEMPLARS = [
    "Q: Ming has 3 pens and buys 4 more. How many pens in total?\n"
    "A: He originally had 3, and after buying 4 it is 3 + 4 = 7 pens. The final answer is 7.",
    "Q: A bookshelf had 9 books; 3 were taken. How many remain?\n"
    "A: Taken means subtract, 9 - 3 = 6. The final answer is 6.",
]


def build_prompt(question, exemplars, tail=""):
    """Concatenate exemplars and the problem to be answered into one prompt; tail can hold a lead-in.

    exemplars: list of exemplar strings; question: the problem to answer.
    The prompt ends with "A:" so the model continues from there.
    """
    parts = exemplars + ["Q: " + question + "\n" + tail + "A:"]
    return "\n\n".join(parts)


print("standard prompt example:")
print(build_prompt(QUESTIONS[0], STANDARD_EXEMPLARS))
print("\nchain-of-thought prompt example:")
print(build_prompt(QUESTIONS[0], COT_EXEMPLARS))


The largest difference between the two prompts is the output path. Under the standard prompt, the model emits the final number in one step; under the chain-of-thought prompt, the model first emits step-by-step calculation, then closes with "The final answer is". Below we implement a parser that extracts the answer number from both kinds of output, then run each once with llm_client. When no API key is configured, the run automatically enters the live-API demo: the model output is placeholder text, and what is exercised is the full parse-and-compare pipeline.

In [ ]:
import os
import re
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()


def parse_answer(text):
    """Extract the answer number from model output; return None if parsing fails.

    Prefer the fixed close "The final answer is N", then "The answer is N",
    and finally fall back to the last integer in the text.
    """
    for pat in [r"The final answer is\s*(\d+)", r"The answer is\s*(\d+)", r"A:\s*(\d+)"]:
        m = re.search(pat, text)
        if m:
            return int(m.group(1))
    nums = re.findall(r"\d+", text)
    return int(nums[-1]) if nums else None


def run_prompt(prompt):
    """Call the model and return the raw output text, using greedy decoding."""
    return client.chat([{"role": "user", "content": prompt}], temperature=0.0)


for label, exemplars in [("standard prompt", STANDARD_EXEMPLARS), ("chain of thought", COT_EXEMPLARS)]:
    print("=" * 22, label, "=" * 22)
    correct = 0
    for q, truth in zip(QUESTIONS, TRUTH):
        out = run_prompt(build_prompt(q, exemplars))
        ans = parse_answer(out)
        if ans == truth:
            correct += 1
        verdict = "hit" if ans == truth else ("no number parsed" if ans is None else "miss")
        print("problem:", q)
        print("output:", out.replace("\n", " ")[:70])
        print(f"parsed {ans} / true {truth} -> {verdict}\n")
    print(f"correct {correct}/{len(QUESTIONS)}\n")

if False:
    print("Live-API demo: model output is placeholder text; the hit counts above do not represent real reasoning ability.")


Under a live API, the two outputs form a contrast: the chain-of-thought answer shows intermediate steps, while the standard answer has only the result. The parser reduces both to a final answer number; later experiments care only about that number. Under the live-API demo both paths are placeholder text and do not show a difference in steps; a real model is needed to observe a full reasoning chain.


The previous section showed that chain-of-thought helps. This section identifies which part that gain comes from. A direct method is to replace the intermediate steps with another form and see whether the effect remains. This experimental method of "removing or replacing one part and observing the change" is called ablation. The CoT paper ran a set of ablations: replacing intermediate steps with pure equations, with ellipses, or moving them after the answer; in all three variants the gain disappeared. The only remaining variable is "writing the steps in natural language, in order". Below we first reproduce that conclusion with a hand-calculated toy model, then implement a decoding method that "tries several times and votes".

First we build a sense of scale, turning "whether chain-of-thought helps" into two numbers that can be compared. Suppose a problem needs L sequential steps. A model that answers directly is correct in one attempt with probability p_direct; a step-by-step model is correct on each step with probability p_step, and the final answer is correct only if all L steps are correct, with probability p_step to the power L. Taking p_direct = 0.25, p_step = 0.80, L = 3, we first compute the two numbers.

Each of the three ablations removes part of the intermediate steps. Equation only: the exemplar has only the final formula, and the model still has to compute in one shot. Dots only: steps are replaced by ellipses, which only hints that "more computation is allowed", not how to compute. Reasoning after the answer: the final answer is given first and reasoning is filled in after; when the model generates the answer it cannot use the steps. All three variants return to "answer in one shot", so their probability is p_direct.

In [ ]:
p_direct = 0.25
p_step = 0.80
L = 3
p_cot = p_step ** L

print(f"direct answer: {p_direct:.3f}")
print(f"step-by-step: each step {p_step:.2f}, all {L} steps correct = {p_step} ** {L} = {p_cot:.3f}")
print(f"gap: {p_cot - p_direct:.3f}")


In [ ]:
def accuracy_direct(n_trials, p_direct, seed):
    """Direct answer: one attempt, correct with probability p_direct."""
    rng = np.random.default_rng(seed)
    return float((rng.random(n_trials) < p_direct).mean())


def accuracy_steps(n_trials, p_step, L, seed):
    """Step-by-step: all L steps must be correct; each step is correct with probability p_step."""
    rng = np.random.default_rng(seed)
    steps = rng.random((n_trials, L)) < p_step
    return float(steps.all(axis=1).mean())


import matplotlib.pyplot as plt

n_trials = 4000
acc = {
    "baseline": accuracy_direct(n_trials, p_direct, 1),
    "equation only": accuracy_direct(n_trials, p_direct, 2),
    "dots only": accuracy_direct(n_trials, p_direct, 3),
    "answer first": accuracy_direct(n_trials, p_direct, 4),
    "chain of thought": accuracy_steps(n_trials, p_step, L, 5),
}

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(list(acc), list(acc.values()), color=["#999999"] * 4 + ["#1f77b4"])
ax.set_ylabel("accuracy")
ax.set_title("Ablation of the intermediate steps")
ax.set_ylim(0, 1)
for i, (k, v) in enumerate(acc.items()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print("Key observation: the three ablations sit on the baseline; only chain of thought is clearly higher.")


**Why writing out the steps helps**

The ablation confirms that the key variable is "writing the steps in natural language, in order". This section explains the three mechanisms behind that.

The first mechanism is moving intermediate results out of the model and into the text. When answering directly, the intermediate result 2×3=6 exists only inside the model; the model has to compute 6, then carry 6 to add 5, all by relaying internal state, which is easy to lose. Step-by-step answering writes 6 into the text, so when the model generates the next step it can read it and does not have to remember it. Putting intermediate results outside, where the model can read them at any time, is called externalizing working memory.

The second mechanism is task decomposition. When finishing in one shot, every step can go wrong, and accuracy is the product of all steps. We already computed this: one shot is 0.25; step-by-step at 0.8 per step, three steps all correct is about 0.512. Step-by-step replaces "get one large computation right in one shot" with "get a small computation right three times", so both the difficulty of a single step and the surface for errors shrink.

The third mechanism is coherence. A language model tends to stay consistent with the preceding text. After a step-by-step model writes "5 + 6 = 11", the next step is unlikely to write "The final answer is 3", because 3 is incoherent with the 11 already written. Intermediate steps become an anchor that constrains later output.

To be clear, chain-of-thought cannot create ability from nothing. What it elicits is reasoning already in the weights; it only changes the calling mode from "one step" to "many steps". That is what was said earlier: some reasoning ability was already there, and the prompt format decides whether it can be called.

One chain-of-thought decode walks a single reasoning path; an error at one step is carried to the end. The improvement is to let the model walk several paths: from the same prompt, sample several times to obtain several answers, each of which computes a final answer, then count which answer appears most often; the most frequent wins. Several paths agreeing on the same answer is called self-consistency decoding. Why it works, in intuition: a multi-step reasoning problem usually has several different correct paths that all lead to the same answer, so correct paths meet at the answer; error paths each fail in their own way and rarely point at the same wrong number. We first compute by hand in a two-answer setting, then simulate with several wrong answers.

In [ ]:
from math import comb

p = 0.7
K = 3
# Two-class case: a strict majority of correct votes is required
maj = sum(comb(K, i) * p ** i * (1 - p) ** (K - i)
          for i in range(K // 2 + 1, K + 1))
print(f"single-sample success probability {p:.2f}")
print(f"{K}-path two-class majority-vote success probability {maj:.3f}")


**Five sampled paths computed by hand**

The mechanism of majority vote can be computed directly on five paths. The problem is "Mom bought 4 bags of apples, 6 per bag. After eating 5, how many are left?", and the true answer is 19. The same prompt is sampled five times, and the model gives five different outputs:

In [ ]:
def majority_accuracy(p, K, W, n_trials, seed):
    """Simulate majority vote: each path is correct with probability p, otherwise it draws uniformly from W wrong answers."""
    rng = np.random.default_rng(seed)
    hits = 0
    for _ in range(n_trials):
        correct = rng.random(K) < p
        wrong = rng.integers(1, W + 1, size=K)
        votes = np.where(correct, 0, wrong)
        counts = np.bincount(votes, minlength=W + 1)
        hits += counts.argmax() == 0
    return hits / n_trials


K_grid = range(1, 41)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for p in [0.5, 0.6, 0.7, 0.8]:
    acc = [majority_accuracy(p, K, W=8, n_trials=2000, seed=int(p * 100))
           for K in K_grid]
    ax.plot(list(K_grid), acc, label=f"p = {p:.1f}")
ax.axhline(1 / 8, color="gray", ls="--", lw=1)
ax.set_xlabel("number of sampled paths K")
ax.set_ylabel("majority vote accuracy")
ax.set_title("Majority vote over sampled paths")
ax.legend()
plt.tight_layout()
plt.show()

print("Key observation: as long as a single path's success probability is above 1/W, vote accuracy approaches 1 quickly with K;")
print("error paths are spread over several error classes and almost never pile onto the same answer.")


Majority vote assumes that "the same wording" counts as the same vote. In free-text answers, the correct wording often has spelling variation; banana written as bananna is common. Exact match treats those two writings as two answers, so the vote is split and may lose to a wrong answer that happens to pile up.

To fix this, there has to be a quantity for "how alike two writings are". The smallest number of insertions, deletions, and substitutions needed to change one string into another is called edit distance. banana and bananna differ by one letter, so the distance is 1; banana and grape require a full rewrite, so the distance is large. The improved method is: merge writings whose edit distance is very small into a group, called a cluster, and vote by the cluster's total count.

Below we implement edit distance and cluster voting from scratch. The test data is synthetic, i.e. generated by a program to simulate real sampled answers; such data is called synthetic data. On synthetic data with spelling noise, we compare cluster voting with exact voting.

**Different writings split the vote**

Majority vote requires "the same answer" to count as the same vote. Free-text answers have no fixed spelling; the correct banana may be written bananna or even banan. Below we compute five paths by hand; the true answer is banana:

```text
path 1: banana
path 2: bananna
path 3: banan
path 4: grape
path 5: grape
```

Counting by exact match first: banana 1 vote, bananna 1 vote, banan 1 vote, grape 2 votes. grape has the most votes, so a wrong answer wins. The problem is that the three correct writings have 1 vote each, split by spelling; grape happens to appear twice and becomes the majority.

After clustering by edit distance and then voting: banana and bananna have distance 1, banana and banan have distance 1, banan and bananna have direct distance 2, but they are connected through banana, so the three writings still join the same cluster. The cluster totals 3 votes, more than grape's 2. The representative answer is the most frequent writing in the cluster, banana, which is correct.

The two votes, before and after clustering, turn a wrong result into a right one. The only extra information is one rule: strings whose edit distance is at most 1 are treated as the same answer. The object of voting is thereby upgraded from "a string" to the abstract notion "different writings of the same answer".

In [ ]:
from collections import Counter


def edit_distance(a, b):
    """Edit distance from string a to b (insertion, deletion, and substitution each count as one)."""
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[-1]


def cluster_vote(samples, thresh=1):
    """Cluster by edit distance, then vote; return the representative answer.

    Treat answers whose edit distance is at most thresh as the same cluster (an edge), use a union-find
    to find all connected components, and take a component's weight as the sum of member frequencies;
    take the highest-weight component, and take its most frequent member as the representative answer.
    """
    counter = Counter(samples)
    words = list(counter)
    parent = list(range(len(words)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if edit_distance(words[i], words[j]) <= thresh:
                union(i, j)

    comp_weight = {}
    for i, w in enumerate(words):
        r = find(i)
        comp_weight[r] = comp_weight.get(r, 0) + counter[w]

    best_r = max(comp_weight, key=comp_weight.get)
    members = [words[i] for i in range(len(words)) if find(i) == best_r]
    return max(members, key=counter.get)


def typo(word, rng):
    """Generate a variant of word at edit distance one; with probability 0.6 return the original word."""
    if rng.random() < 0.6:
        return word
    i = rng.integers(0, len(word))
    if rng.random() < 0.5:
        return word[:i] + word[i + 1:]
    return word[:i] + ("z" if word[i] != "z" else "q") + word[i + 1:]


WORDS = ["banana", "grape", "peach", "melon", "lemon"]
WRONGS = ["apple", "plum", "cherry", "kiwi", "mango", "fig"]


def simulate_batch(n_questions, K, p_correct, seed):
    """Generate sampled answers for a batch of questions; return accuracies of three strategies.

    Each path outputs a variant of the correct word with probability p_correct, otherwise an unrelated wrong word.
    """
    rng = np.random.default_rng(seed)
    single = exact = clustered = 0
    for _ in range(n_questions):
        truth = WORDS[rng.integers(0, len(WORDS))]
        samples = []
        for _ in range(K):
            if rng.random() < p_correct:
                samples.append(typo(truth, rng))
            else:
                samples.append(WRONGS[rng.integers(0, len(WRONGS))])
        single += samples[0] == truth
        exact += Counter(samples).most_common(1)[0][0] == truth
        clustered += cluster_vote(samples) == truth
    n = float(n_questions)
    return single / n, exact / n, clustered / n


print("edit-distance examples: banana vs bananna =", edit_distance("banana", "bananna"),
      ", banana vs grape =", edit_distance("banana", "grape"))

p_correct = 0.6
K = 9
res = simulate_batch(3000, K, p_correct, 7)
for name, v in zip(["single sample", "exact majority vote", "edit-distance cluster vote"], res):
    print(f"{name}: accuracy {v:.3f}")
print("Key observation: cluster voting merges spelling variants of the correct word, so accuracy is higher than exact voting.")


The gain from voting depends on diversity of sampled paths: the paths must differ, or voting has no meaning. One knob that controls diversity is called temperature. Higher temperature makes model sampling more random, so paths branch more easily; when temperature approaches 0, sampling degenerates into always choosing the highest-probability word. That generation method is called greedy decoding; K paths become identical, and voting is empty. Temperature is not "higher is better": the flatter the distribution, the lower each path's individual success probability. Below we write both effects as functions of temperature with a sampler of our own, and observe how majority-vote accuracy changes with temperature.

**How temperature makes paths branch**

When the model predicts the next token, it assigns each possible token a raw score called logits; a higher score means a stronger preference for that token. To turn scores into probabilities, take the exponential of each score and divide by the sum of all scores so that probabilities add to 1; that process is called softmax. Temperature T is inserted before softmax: at sampling time, divide logits by T, then apply softmax:

$$P(\text{token}_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

Larger T makes the divided values closer, so the distribution is flatter; smaller T magnifies differences, so the distribution is sharper. A three-class example: suppose three tokens have logits [2.0, 1.0, 0.0]:

| T | P(token1) | P(token2) | P(token3) | shape of the distribution |
|:---|:---|:---|:---|:---|
| 0.5 | 0.867 | 0.117 | 0.016 | almost always token1 |
| 1.0 | 0.665 | 0.245 | 0.090 | biased, but with room |
| 2.0 | 0.506 | 0.307 | 0.186 | near-uniform, token3 also has a chance |

Take T=1 and compute one row by hand. The exp values are $[e^2, e^1, e^0]$ = [7.39, 2.72, 1], sum 11.11, divide to get [0.665, 0.245, 0.090]. T=0.5 is equivalent to multiplying logits by 2 before softmax; exp becomes $[e^4, e^2, e^0]$, the gap is magnified, and token1's probability approaches 0.87. T=2 is equivalent to dividing by 2, and the three probabilities converge.

Connecting temperature to self-consistency voting: when T approaches 0, every step picks the highest-probability token, five samples walk five nearly identical paths, and voting equals not voting. After T rises, low-probability tokens also have a chance to be picked, paths start to branch, which is the diversity voting needs. Temperature is not "higher is better": the flatter the distribution, the more the model resembles random guessing, and each path's individual success probability falls. The role of temperature is to find a middle value where "paths branch enough, but not at random"; in the simulation the empirical values for PaLM and UL2 are 0.5 to 0.7.

In [ ]:
def temperature_accuracy(temp, K, p_good, tau, div_scale, W, n_trials, seed):
    """Simulate the effect of temperature on majority vote.

    Rising temp raises path diversity and lowers single-path quality at the same time;
    the two effects are each modeled exponentially.
    """
    rng = np.random.default_rng(seed)
    p = p_good * np.exp(-temp / tau)          # single-path success probability
    div = 1.0 - np.exp(-temp / div_scale)     # path diversity
    hits = 0
    for _ in range(n_trials):
        if rng.random() < div:
            correct = rng.random(K) < p
            wrong = rng.integers(1, W + 1, size=K)
            votes = np.where(correct, 0, wrong)
            counts = np.bincount(votes, minlength=W + 1)
            hits += counts.argmax() == 0
        else:
            hits += rng.random() < p_good     # low-temperature degeneration: all K paths equal the greedy path
    return hits / n_trials


temp_grid = np.linspace(0.0, 2.0, 11)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for K in [1, 5, 40]:
    acc = [temperature_accuracy(t, K, 0.6, 1.5, 0.6, 4, 1500, 11)
           for t in temp_grid]
    ax.plot(temp_grid, acc, marker="o", label=f"K = {K}")
ax.set_xlabel("temperature")
ax.set_ylabel("accuracy")
ax.set_title("Temperature and the number of sampled paths")
ax.legend()
plt.tight_layout()
plt.show()

print("Key observation: more paths make accuracy less sensitive to temperature; when temperature is too high, single-path quality")
print("falls below 1/W, and majority vote fails with it. In practice PaLM and UL2 take sampling temperature in 0.5 to 0.7.")


The vote distribution over several paths is itself information. Intuitively, several paths pointing at the same answer means the model is confident in that answer; paths each saying something different means the model is unsure. To quantify that confidence: look at how many votes the top answer has, compute its share; the closer the share is to 1, the more consistent the paths. That share is called a consistency score. We synthesize a mix of easy and hard problems, check whether the consistency score correlates with "whether majority vote is correct", then demonstrate a simple policy: if consistency is below a threshold, skip the problem rather than answer. In an Agent framework, this signal is exactly the basis on which the model judges "when it should not decide on its own".

In [ ]:
def consistency_correctness(n_questions, K, W, seed):
    """Simulate a mix of easy and hard problems; return arrays (max vote share, whether majority is correct)."""
    rng = np.random.default_rng(seed)
    fractions, corrects = [], []
    for _ in range(n_questions):
        p = 0.90 if rng.random() < 0.45 else 0.12   # about 45% easy, 55% hard
        ok = rng.random(K) < p
        wrong = rng.integers(1, W + 1, size=K)
        votes = np.where(ok, 0, wrong)
        counts = np.bincount(votes, minlength=W + 1)
        fractions.append(counts.max() / K)
        corrects.append(counts.argmax() == 0)
    return np.array(fractions), np.array(corrects)


K, W = 20, 6
fractions, corrects = consistency_correctness(4000, K, W, 41)
r = np.corrcoef(fractions, corrects)[0, 1]
acc_all = corrects.mean()
acc_accept = corrects[fractions >= 0.5].mean()

edges = np.linspace(0, 1, 21)
xf, yc = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    mask = (fractions >= lo) & (fractions < hi)
    if mask.sum() > 10:
        xf.append(fractions[mask].mean())
        yc.append(corrects[mask].mean())

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(xf, yc, marker="o")
ax.set_xlabel("max vote fraction (consistency)")
ax.set_ylabel("probability majority is correct")
ax.set_title("Consistency as an uncertainty signal")
plt.tight_layout()
plt.show()

print(f"correlation of consistency score with correctness: {r:.3f}")
print(f"accuracy on all problems {acc_all:.3f}; accuracy only on problems with consistency ≥ 0.5 {acc_accept:.3f}")
print("Key observation: the more concentrated the votes, the more likely the answer is correct; consistency can serve as a threshold for refusing to answer.")


The previous sections all used arithmetic problems. The CoT paper has another result, on a different kind of task that involves no calculation, only concatenating and transforming symbols by rule, called a symbol task. The phenomenon it observes is called length extrapolation: exemplars give only short inputs, and at test time the input grows longer; the question is whether the model can keep getting it right. The task itself is simple: take the last letter of each word in a name and concatenate them. The input Mary Jane Smith, step by step, yields y, e, h, output yeh. Few-shot exemplars use only short 2-word names; the model learns the rule "process word by word", and can reuse it on 3-word and 4-word names. A model that answers directly only memorizes the short exemplars and fails as soon as length grows. Below we simulate step-by-step and direct strategies with rules, and compare accuracy under length extrapolation.

**Last-letter task: a symbol example that can be computed**

The CoT paper's length-extrapolation experiment uses last-letter concatenation: given a name, take the last letter of each word and concatenate them. Mary Jane Smith step by step yields y, e, h, output yeh. Few-shot exemplars give only 2-word names; from the exemplars the model learns the rule "for each word, output its last letter". That rule does not depend on the number of words, so 3-word and 4-word names can reuse it word by word. A model that answers directly does not learn this rule; it only memorizes the short exemplars and can only guess on longer names.

This difference can be computed with two probability models. First, direct answering: it has to process the whole string of words at once; the more words, the larger the chance of an error anywhere in the middle, so accuracy decays exponentially with length. We describe that decay by $p_{\text{direct}}(L) = 0.7\, e^{-0.8(L-2)}$. Then step-by-step answering: each word is correct with probability 0.9, all L words must be correct, so accuracy is $0.9^L$. Substituting each length:

| Length (words) | Direct answer | Step-by-step |
|:---|:---|:---|
| 2 | 0.70 | 0.81 |
| 3 | 0.31 | 0.73 |
| 4 | 0.14 | 0.66 |
| 5 | 0.06 | 0.59 |

As length goes from 2 to 5, direct-answer accuracy falls from 0.70 to 0.06, while step-by-step only falls from 0.81 to 0.59. The difference comes from where the task sits in the two strategies: direct answering treats "how many words to process" as part of task difficulty, so more words mean more errors; step-by-step splits the task into independent "take the last letter" micro-steps, each step's difficulty does not change with length, and accuracy falls with length only because the number of steps grows.

This example shows the source of chain-of-thought extrapolation: the step-by-step strategy turns "length extrapolation" into "doing a short task that has been seen, at every step". The model does not need to have seen a 5-word name; it only needs to get "take the last letter" right five times in a row.

In [ ]:
def direct_accuracy(length, p0, decay, seed):
    """Direct answer: accuracy decays exponentially with length, simulating failure to generalize to long examples."""
    rng = np.random.default_rng(seed)
    p = p0 * np.exp(-decay * (length - 2))
    return float((rng.random(3000) < p).mean())


def stepwise_accuracy(length, p_step, seed):
    """Step-by-step: each step is correct with probability p_step; all must be correct to score."""
    rng = np.random.default_rng(seed)
    steps = rng.random((3000, length)) < p_step
    return float(steps.all(axis=1).mean())


lengths = [2, 3, 4, 5]
direct = [direct_accuracy(L, 0.7, 0.8, 21) for L in lengths]
stepwise = [stepwise_accuracy(L, 0.9, 22) for L in lengths]

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(lengths, direct, marker="o", label="direct answer")
ax.plot(lengths, stepwise, marker="s", label="step-by-step")
ax.set_xlabel("number of words (out of distribution)")
ax.set_ylabel("accuracy")
ax.set_title("Length extrapolation on symbol tasks")
ax.set_xticks(lengths)
ax.legend()
plt.tight_layout()
plt.show()

print("direct-answer accuracy at lengths", lengths, ":", [f"{v:.2f}" for v in direct])
print("step-by-step accuracy at lengths", lengths, ":", [f"{v:.2f}" for v in stepwise])
print("Key observation: direct answering collapses under length extrapolation; step-by-step stays robust after splitting long examples into short steps.")


The previous experiments keep showing the same phenomenon: small models get no gain from chain-of-thought, or even regress; after model scale crosses a threshold, the gain appears suddenly. This "suddenly strong after a threshold" phenomenon is called emergence: ability stays near chance in one scale band, then jumps well above chance after a critical scale. The emergence paper (Wei et al., 2022) collected such abilities into a list; chain-of-thought and self-consistency decoding are both on it.

There is a point worth pressing: the jump that is seen may be caused only by the metric. A metric is the way of scoring the model's answer; different scoring methods yield different curve shapes. One metric that only distinguishes right from wrong is a binary metric: the whole answer must match the reference exactly to count as correct, and a near miss counts as zero; exact match is a typical example. Another metric gives partial credit, for example per-token accuracy: count token by token, take the fraction correct, and partial correctness still scores. Yi et al. (2204.07646) redrew the same curves with metrics such as per-token accuracy, found that the curves became smooth and predictable, and on that basis argued that "emergence" is an artifact produced by binary metrics. Both sides agree that underlying ability grows smoothly; the disagreement is which metric should be used to read task-level accuracy. Below we replay this debate on synthetic data.

Construct an underlying ability that grows smoothly with scale: write the answer as a 5-token sequence, each token independently correct, with single-token accuracy rising smoothly from 0.2 to 0.9. Read it with two metrics: per-token accuracy equals single-token accuracy directly, a smooth straight line; exact match requires all 5 tokens correct, so accuracy is single-token accuracy to the 5th power, and a clear jump appears near single-token accuracy 0.5. The same underlying ability, two readings.


**The same ability, two readings: five scale points computed by hand**

Compute this toy ability at five scale points under both metrics. Single-token accuracy p grows linearly with scale; take five levels p = 0.41, 0.48, 0.55, 0.62, 0.69, each rising by only 0.07:

| Single-token accuracy p | Per-token accuracy = p | Exact match = $p^5$ |
|:---|:---|:---|
| 0.41 | 0.410 | 0.012 |
| 0.48 | 0.480 | 0.025 |
| 0.55 | 0.550 | 0.050 |
| 0.62 | 0.620 | 0.092 |
| 0.69 | 0.690 | 0.156 |

Per-token accuracy is p itself; the five points form a straight line, rising uniformly. Exact match is p to the 5th power; the five points rise from 0.012 to 0.156, steeper toward the right: p from 0.48 to 0.55 rises only 14.6%, while exact match from 0.025 to 0.050 nearly doubles; p from 0.55 to 0.62 rises only 12.7%, while exact match rises another 82%.

Underlying ability rose only 0.07 per step, with no discontinuity; the apparent jump comes from the nonlinear metric p to the 5th power amplifying a small increment. More precisely, exact match is a binary metric: all 5 tokens must be correct to score, one error zeroes the whole sentence, and partial credit is wiped out; per-token accuracy keeps partial credit. The same underlying ability, different metrics, different shapes. That is the full meaning of "emergence is an artifact of the metric" — the underlying is smooth, and the metric amplifies smoothness into steepness.

In [ ]:
s = np.linspace(0, 1, 200)
p_tok = 0.2 + 0.7 * s            # single-token accuracy: smooth linear growth
L = 5
exact_match = p_tok ** L          # exact match: score only if all are correct
per_token = p_tok                 # per-token: partial credit also scores

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(s, per_token, label="per-token accuracy")
ax.plot(s, exact_match, label="exact match (5 tokens)")
ax.set_xlabel("scale")
ax.set_ylabel("accuracy")
ax.set_title("The same ability under two metrics")
ax.legend()
plt.tight_layout()
plt.show()

print("Key observation: the exact match curve shows a phase-transition shape, the per-token curve is smooth;")
print("underlying ability has no discontinuity; the jump comes from the metric wiping out all partial credit.")


The same mechanism is clearer on multi-step tasks. Suppose per-step success probability p grows smoothly, and final accuracy is p to the power L. The larger L, the flatter the low-p band is pressed, and the steeper the jump. That is also one reason chain-of-thought only takes effect on large enough models: a small model's multi-step accuracy is pressed near zero, and only after crossing a threshold does it begin to show. Plot the three curves L = 1, 3, 5.


**The more steps, the stronger the amplification**

The same compounding is more extreme on longer tasks. Let per-step success probability be p, and final accuracy $p^L$. Fix several levels of p, and look at final accuracy under different L:

| p | L=1 (one-shot) | L=3 | L=5 |
|:---|:---|:---|:---|
| 0.5 | 0.500 | 0.125 | 0.031 |
| 0.6 | 0.600 | 0.216 | 0.078 |
| 0.7 | 0.700 | 0.343 | 0.168 |
| 0.8 | 0.800 | 0.512 | 0.328 |

Look at the L=5 column. p from 0.5 to 0.6 rises only 0.1, while final accuracy from 0.031 to 0.078 becomes 2.5 times the original; p from 0.7 to 0.8 rises another 0.1, while final accuracy from 0.168 to 0.328 nearly doubles. The same 0.1 increment is amplified more on the large-p side.

This explains two observations. First, the more steps, the closer final accuracy in the low-p band is pressed to zero, and the more the curve looks like a jump of "either near zero, or up"; that is why emergence looks steeper when L is large. Second, chain-of-thought only takes effect on large enough models: a small model's per-step success probability is low, and after L-fold multiplication accuracy is pressed near chance, so no effect is visible; after the model grows, per-step probability rises, and the product turns from near zero to clearly above chance.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for L in [1, 3, 5]:
    ax.plot(s, p_tok ** L, label=f"{L} steps: p^L")
ax.set_xlabel("per-step accuracy (smooth)")
ax.set_ylabel("task accuracy")
ax.set_title("Compounding single-step ability into task accuracy")
ax.legend()
plt.tight_layout()
plt.show()

print("Key observation: the same smooth single-step ability, with more steps, makes the final accuracy curve closer to a jump.")


This debate leaves an operational lesson: underlying ability grows on a smooth curve, while the task-level accuracy users care about is determined by a compounding process; the apparent jump is that compounding amplified by a binary metric. The emergence paper supplies the other side with evidence the critics did not address. During training there is a quantity that measures the gap between the model's prediction and the true answer, called loss; cross-entropy is a common one. Read with cross-entropy, loss in the small-scale band does fall steadily, showing that ability is accumulating; and classification tasks measured by exact match also show emergence. Those two points show that the metric cannot fully explain the jump. A more careful reading splits the difference: ability accumulates smoothly, and the metric decides how it is seen.

The previous sections all elicited reasoning at the prompt level: the user constructs exemplars, tunes temperature, and votes; the ability ceiling is constrained by the prompt, and a new problem often needs a new prompt design. Reasoning models from 2024 onward (OpenAI o1, DeepSeek-R1) take another road: move "multi-step reasoning" into the training stage, and let the model learn to think internally on its own. The training method is called reinforcement learning: score the model's answer, encourage good answers and penalize bad ones, and through repeated practice the model learns to write longer, more accurate chains of thought. After such training, multi-step reasoning is already internalized as part of the weights; at inference a short instruction is enough, and the chain of thought is no longer provided by the prompt but generated by the model itself.

The difference between the two generations can be read side by side:

| Dimension | Prompt-elicitation era | Reasoning-model era |
|:---|:---|:---|
| Where reasoning comes from | Intermediate steps in few-shot exemplars | Reinforcement learning at training time |
| Whether weights change | No; prompt and decoding-side changes only | Yes; learned in post-training |
| Where paths come from | The user writes exemplars as demonstration | The model generates the chain of thought itself |
| Reasoning budget | Number of samples, temperature | Number of hidden thinking tokens |
| Uncertainty | Vote share from self-consistency | Model-reported confidence |

The two eras are not opposed. Prompt techniques remain useful on reasoning models, and self-consistency voting as a "knowing that it does not know" signal is still an important Agent component: take a batch of samples, first see whether the votes concentrate, adopt only if they concentrate, and refuse to answer if they do not. The next lecture enters mathematical reasoning (AlphaProof) and pushes this road to the level of proofs.


## Summary

This lecture walked the path of "how reasoning ability is called":

- [ ] Reasoning ability comes from pretraining; the prompt format decides whether it can be called; a standard prompt is only a lower bound on ability
- [ ] Changing exemplars to ⟨problem, reasoning steps, answer⟩ lets chain-of-thought elicit multi-step reasoning, with no training
- [ ] Ablation shows the key variable is "writing the steps in natural language, in order"; equations, ellipses, and post-hoc reasoning bring no gain
- [ ] Repeated sampling plus majority vote aggregates several paths at the answer; error paths rarely meet
- [ ] Edit-distance cluster voting can merge different writings of the same answer, more stable than exact voting
- [ ] Temperature balances path diversity and quality; more paths are less sensitive to temperature
- [ ] Consistency score correlates with correctness, and can serve as a "knowing whether it knows" confidence signal
- [ ] Word-by-word rules on symbol tasks let chain-of-thought extrapolate on longer sequences; direct answering collapses
- [ ] The same underlying ability looks like emergence or smoothness under different metrics; the metric affects the conclusion


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

The three problems correspond to chain-of-thought parsing and ablation, self-consistency aggregation, and emergence metrics. Each problem has a code snippet with blanks; after filling them, run the asserts to self-check.


In [ ]:
import re

# Exercise 1: complete the answer-parsing regex and the multi-step accuracy formula
# Blank 1: match a close such as "The final answer is N"; fill in r"The final answer is\s*(\d+)"
PATTERN = r"The final answer is\s*(\d+)"

def parse_ans(text):
    m = re.search(PATTERN, text)
    return int(m.group(1)) if m else None

assert parse_ans("The answer is 5. Step by step we get 5. The final answer is 11.") == 11
assert parse_ans("The final answer is 19.") == 19

# Blank 2: all L steps must be correct to score; fill in p_step ** L
p_step, L = 0.8, 3
p_cot = p_step ** L
assert p_cot > 0.5, "after step-by-step, accuracy should be clearly above one-shot 0.25"
print(f"parser and formula are correct: probability all 3 steps are correct {p_cot:.3f}, above direct 0.25.")
# Hint: the regex captures only the integer after "The final answer is"; multi-step accuracy is the product of per-step probabilities.


In [ ]:
# Exercise 2: complete majority vote and edit distance
# Blank 1: take the most frequent answer; fill in counts.argmax()
def majority(answers, W=8):
    votes = np.array(answers)
    counts = np.bincount(votes, minlength=W + 1)
    return int(counts.argmax())

assert majority([3, 3, 5, 3, 5]) == 3
assert majority([1, 2, 2]) == 2

# Blank 2: substitution costs 1 only when characters differ; fill in a[i - 1] != b[j - 1]
def edit_distance2(a, b):
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[-1]

assert edit_distance2("banana", "bananna") == 1
assert edit_distance2("banana", "grape") > 1
print("Majority vote and edit distance are implemented correctly; different writings of the same answer merge inside a cluster.")
# Hint: majority vote counts frequency; the substitution branch of edit distance adds one only when characters differ.


In [ ]:
# Exercise 3: complete both metrics and implement an emergence detector
# Blank 1: exact match requires all L tokens correct; fill in p ** L
# Blank 2: per-token accuracy also scores partial credit; fill in p
def metrics(p, L):
    exact = p ** L
    per_token = p
    return exact, per_token

# Emergence detector: if the back-half slope is more than a threshold times the front-half slope, call it emergence
def detect_emergent(x, y, thr=2.0):
    half = len(x) // 2
    slope_front = (y[half] - y[0]) / (x[half] - x[0])
    slope_back = (y[-1] - y[half]) / (x[-1] - x[half])
    return slope_back / (slope_front + 1e-9) > thr

s = np.linspace(0, 1, 200)
p_tok = 0.2 + 0.7 * s
exact, per_tok = metrics(p_tok, 5)
assert detect_emergent(s, exact), "the exact match curve should be judged emergent"
assert not detect_emergent(s, per_tok), "the per-token curve should be judged smooth"
print("Metric check passed: the same underlying ability looks emergent under exact match and smooth under per-token.")
# Hint: compute mean slope on the front half and the back half; per-token is a straight line, so the two slopes match.


## References

- [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903) (Wei et al., 2022) — the anchor of this lecture: few-shot chain-of-thought prompting, ablations, and scale emergence
- [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171) (Wang et al., 2022) — sample-and-marginalize: sample several paths, then majority vote
- [Emergent Abilities of Large Language Models](https://arxiv.org/abs/2206.07682) (Wei et al., 2022) — definition of emergent abilities, a survey of evidence, and self-critique
- [Are Emergent Abilities of Large Language Models a Mirage?](https://arxiv.org/abs/2204.07646) (Yi et al., 2022) — the opposing view: emergence as an artifact of discontinuous metrics; contrast with section 3 of this lecture
- [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168) (Cobbe et al., 2021) — the GSM8K dataset and the "finetune plus verifier" baseline
- [Large Language Models are Zero-Shot Reasoners](https://arxiv.org/abs/2205.11916) (Kojima et al., 2022) — zero-shot chain-of-thought: Let's think step by step
- [Least-to-Most Prompting Enables Complex Reasoning](https://arxiv.org/abs/2205.10625) (Zhou et al., 2022) — an advanced decomposition strategy for chain-of-thought
- [Show Your Work: Scratchpads for Intermediate Computation](https://arxiv.org/abs/2112.00114) (Nye et al., 2021) — predicting intermediate computation, already emergent on very small models
- [Finetuned Language Models are Zero-Shot Learners](https://arxiv.org/abs/2109.01652) (Wei et al., 2021) — emergence in FLAN instruction tuning
- [BIG-bench: Beyond the Imitation Game](https://arxiv.org/abs/2206.04615) (Srivastava et al., 2022) — a main source of emergence evidence
